# Lab 28 Kaggle Bootstrap - Real vLLM

Use this notebook for final Lab 28 submission when the requirement is real vLLM on Kaggle.

This notebook starts:

- real `vLLM` on `localhost:8001`
- one FastAPI gateway on `localhost:8000`
- `/v1/...` proxied to vLLM
- `/embed` served by sentence-transformers

The final public ngrok URL is intentionally shared by `VLLM_NGROK_URL` and `EMBED_NGROK_URL` so free ngrok does not replace one tunnel with another.


In [ ]:
# Install runtime dependencies. If Kaggle asks you to restart after install, restart and run again from this cell.
!pip install -q vllm fastapi uvicorn pyngrok sentence-transformers httpx requests mlflow

In [ ]:
# Sanity check GPU. This must show a GPU before vLLM can work.
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
from getpass import getpass

# Paste your ngrok auth token when Kaggle prompts for it. It will not be printed in the notebook.
NGROK_AUTH_TOKEN = getpass("Paste NGROK_AUTH_TOKEN: ").strip()

# Small but real vLLM model for Kaggle T4 reliability. You may raise this after the lab is stable.
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"

VLLM_PORT = 8001
GATEWAY_PORT = 8000
MAX_MODEL_LEN = 1024
GPU_MEMORY_UTILIZATION = 0.60

assert NGROK_AUTH_TOKEN, "Fill in NGROK_AUTH_TOKEN first"


In [ ]:
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

import requests
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
VLLM_LOG_PATH = Path("/kaggle/working/vllm.log")

# Kaggle is more reliable with vLLM's legacy engine on small T4 sessions.
vllm_env = os.environ.copy()
vllm_env["VLLM_USE_V1"] = "0"
vllm_env["TOKENIZERS_PARALLELISM"] = "false"
vllm_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
vllm_env["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"


def tail_log(path=VLLM_LOG_PATH, lines=260):
    if not path.exists():
        print("No vLLM log file yet.")
        return
    content = path.read_text(errors="replace").splitlines()
    print("\n".join(content[-lines:]))


def stop_process(name):
    proc = globals().get(name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=15)
        except subprocess.TimeoutExpired:
            proc.kill()


stop_process("vllm_proc")

base_cmd = [
    sys.executable,
    "-m",
    "vllm.entrypoints.openai.api_server",
    "--host",
    "0.0.0.0",
    "--port",
    str(VLLM_PORT),
    "--model",
    MODEL_NAME,
    "--served-model-name",
    MODEL_NAME,
    "--max-model-len",
    str(MAX_MODEL_LEN),
    "--gpu-memory-utilization",
    str(GPU_MEMORY_UTILIZATION),
    "--dtype",
    "half",
    "--tensor-parallel-size",
    "1",
    "--max-num-seqs",
    "2",
    "--max-num-batched-tokens",
    "1024",
    "--enforce-eager",
    "--trust-remote-code",
]

def start_vllm(extra_args=None):
    cmd = base_cmd + (extra_args or [])
    print("Starting vLLM:", " ".join(cmd))
    log_file = VLLM_LOG_PATH.open("w", encoding="utf-8")
    return subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, text=True, env=vllm_env)


vllm_proc = start_vllm()

health_url = f"http://127.0.0.1:{VLLM_PORT}/v1/models"
last_error = None
for attempt in range(1, 121):
    if vllm_proc.poll() is not None:
        print("vLLM exited with code:", vllm_proc.poll())
        tail_log()
        raise RuntimeError("vLLM exited before becoming healthy")
    try:
        resp = requests.get(health_url, timeout=5)
        if resp.status_code == 200:
            print("vLLM is healthy:", resp.text[:500])
            break
        last_error = f"status={resp.status_code}, body={resp.text[:200]}"
    except Exception as exc:
        last_error = repr(exc)
    if attempt % 6 == 0:
        print(f"Waiting for vLLM... attempt {attempt}/120; last_error={last_error}")
    time.sleep(5)
else:
    tail_log()
    raise TimeoutError(f"vLLM did not become healthy. Last error: {last_error}")

In [ ]:
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, Response
from sentence_transformers import SentenceTransformer
import httpx
import threading
import uvicorn

stop_process("gateway_proc") if "gateway_proc" in globals() else None

gateway_app = FastAPI(title="Lab28 Kaggle Gateway - real vLLM")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)


@gateway_app.get("/healthz")
def healthz():
    try:
        resp = requests.get(f"http://127.0.0.1:{VLLM_PORT}/v1/models", timeout=5)
        vllm_up = resp.status_code == 200
    except Exception:
        vllm_up = False
    return {"status": "ok", "vllm_up": vllm_up, "model": MODEL_NAME}


@gateway_app.post("/embed")
def embed(data: dict):
    texts = data["texts"]
    embeddings = embed_model.encode(texts).tolist()
    return {"embeddings": embeddings}


@gateway_app.api_route("/v1/{path:path}", methods=["GET", "POST"])
async def proxy_vllm(path: str, request: Request):
    target = f"http://127.0.0.1:{VLLM_PORT}/v1/{path}"
    body = await request.body()
    headers = {k: v for k, v in request.headers.items() if k.lower() != "host"}
    try:
        async with httpx.AsyncClient(timeout=180) as client:
            resp = await client.request(
                request.method,
                target,
                params=dict(request.query_params),
                content=body,
                headers=headers,
            )
    except Exception as exc:
        return JSONResponse(
            status_code=502,
            content={"detail": f"vLLM proxy failed: {exc}", "hint": "Check /kaggle/working/vllm.log"},
        )

    forwarded_headers = {
        k: v
        for k, v in resp.headers.items()
        if k.lower() not in {"content-length", "transfer-encoding", "connection"}
    }
    return Response(content=resp.content, status_code=resp.status_code, headers=forwarded_headers)


def run_gateway():
    uvicorn.run(gateway_app, host="0.0.0.0", port=GATEWAY_PORT, log_level="info")


threading.Thread(target=run_gateway, daemon=True).start()

for attempt in range(1, 31):
    try:
        resp = requests.get(f"http://127.0.0.1:{GATEWAY_PORT}/healthz", timeout=5)
        if resp.status_code == 200 and resp.json().get("vllm_up"):
            print("Gateway is healthy:", resp.json())
            break
    except Exception as exc:
        last_error = repr(exc)
    time.sleep(2)
else:
    raise RuntimeError("Gateway did not become healthy")

In [ ]:
# Expose one public URL for both /v1/... and /embed.
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

gateway_tunnel = ngrok.connect(GATEWAY_PORT, "http")
print("Gateway public URL:", gateway_tunnel.public_url)

In [ ]:
headers = {"ngrok-skip-browser-warning": "true"}
base_url = gateway_tunnel.public_url.rstrip("/")

models_resp = requests.get(f"{base_url}/v1/models", headers=headers, timeout=60)
print("vLLM public status:", models_resp.status_code)
print(models_resp.text[:500])
models_resp.raise_for_status()

chat_resp = requests.post(
    f"{base_url}/v1/chat/completions",
    headers=headers,
    json={
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": "Say hello from real vLLM on Kaggle in one short sentence."}],
        "max_tokens": 64,
        "temperature": 0.1,
    },
    timeout=120,
)
print("vLLM chat status:", chat_resp.status_code)
print(chat_resp.text[:700])
chat_resp.raise_for_status()

embed_resp = requests.post(
    f"{base_url}/embed",
    json={"texts": ["hello from lab 28"]},
    headers=headers,
    timeout=60,
)
print("Embedding health status:", embed_resp.status_code)
payload = embed_resp.json()
print("Embedding size:", len(payload["embeddings"][0]))
embed_resp.raise_for_status()

In [ ]:
# Optional MLflow metadata for Lab 28 Integration 6/7 evidence.
import mlflow

mlflow.set_tracking_uri("file:///kaggle/working/mlruns")
with mlflow.start_run(run_name="lab28-real-vllm-serving"):
    mlflow.log_param("model", MODEL_NAME)
    mlflow.log_param("server", "vLLM")
    mlflow.log_param("max_model_len", MAX_MODEL_LEN)
    mlflow.log_param("gpu_memory_utilization", GPU_MEMORY_UTILIZATION)
    mlflow.set_tag("serving_url", gateway_tunnel.public_url)
    mlflow.set_tag("embedding_model", EMBED_MODEL_NAME)

print("MLflow metadata logged under /kaggle/working/mlruns")

In [ ]:
print("Copy these into your local .env file:")
print(f"VLLM_NGROK_URL={gateway_tunnel.public_url}")
print(f"EMBED_NGROK_URL={gateway_tunnel.public_url}")
print(f"MODEL_NAME={MODEL_NAME}")
print("ALLOW_LLM_FALLBACK=false")

## Debug vLLM startup

Run the next cell only if the vLLM startup cell fails.


In [ ]:
# Print vLLM debug log tail
from pathlib import Path
log_path = Path("/kaggle/working/vllm.log")
if log_path.exists():
    lines = log_path.read_text(errors="replace").splitlines()
    print("\n".join(lines[-260:]))
else:
    print("No /kaggle/working/vllm.log found")
